In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约3-5分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q openai-whisper
!pip install -q jieba
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 预下载Whisper模型
import whisper
print('正在下载Whisper small模型...')
model = whisper.load_model('small')
print('Whisper模型就绪！')


# 模块6 第1次课：ASR原理与主流方法本notebook包含：1. ASR问题定义与评估指标（WER、CER）2. ASR经典流程：特征提取→声学模型→语言模型→解码3. CTC（Connectionist Temporal Classification）原理4. Attention-based Seq2Seq + Transformer5. Whisper架构与使用6. ASR与CI研究的关系---

## §1 ASR问题定义**自动语音识别（ASR）**：将语音信号转换为文本。- 输入：音频信号 x(t)- 输出：文本序列 y = [y_1, y_2, ..., y_L]- 目标：找到最可能的文本 y* = argmax P(y|x)**ASR在CI研究中的角色**：- ASR可以作为CI语音处理效果的**客观评估工具**- CI用户最终能否听懂 → 可以用ASR识别准确率来量化- 构建评估闭环：带噪语音 → 处理 → ASR识别 → 计算准确率

## §2 评估指标### WER（Word Error Rate，词错误率）WER = (S + D + I) / N- S：替换（Substitution）- D：删除（Deletion）- I：插入（Insertion）- N：参考文本的词数### CER（Character Error Rate，字符错误率）中文没有空格分词，所以用字符级别的错误率：CER = (S + D + I) / N_chars> **注意**：评估中文ASR必须用CER，不能用WER。

In [ ]:
# WER/CER 计算演示import numpy as npdef levenshtein_distance(ref, hyp):    """计算编辑距离"""    n = len(ref)    m = len(hyp)    dp = np.zeros((n + 1, m + 1), dtype=int)    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref[i-1] == hyp[j-1]:                dp[i][j] = dp[i-1][j-1]            else:                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])    return dp[n][m]def compute_cer(reference, hypothesis):    """计算字符错误率 (CER)"""    # 中文直接按字符拆分    ref_chars = list(reference.replace(" ", ""))    hyp_chars = list(hypothesis.replace(" ", ""))    dist = levenshtein_distance(ref_chars, hyp_chars)    return dist / len(ref_chars) if len(ref_chars) > 0 else 0.0def compute_wer(reference, hypothesis):    """计算词错误率 (WER)"""    ref_words = reference.split()    hyp_words = hypothesis.split()    dist = levenshtein_distance(ref_words, hyp_words)    return dist / len(ref_words) if len(ref_words) > 0 else 0.0# 中文CER示例ref_cn = "今天天气很好"hyp_cn1 = "今天天气很好"  # 完全正确hyp_cn2 = "今天天汽很好"  # 1个替换hyp_cn3 = "今天天气"       # 2个删除hyp_cn4 = "今天天气很好啊" # 1个插入print("=== 中文CER计算示例 ===")for hyp, desc in [(hyp_cn1, "完全正确"), (hyp_cn2, "1个替换"),                   (hyp_cn3, "2个删除"), (hyp_cn4, "1个插入")]:    cer = compute_cer(ref_cn, hyp)    print("  参考: '%s' -> 识别: '%s' (%s) CER=%.2f" % (ref_cn, hyp, desc, cer))print()# 英文WER示例ref_en = "the cat sat on the mat"hyp_en = "the cat sat on mat"print("=== 英文WER计算示例 ===")print("  参考: '%s'" % ref_en)print("  识别: '%s'" % hyp_en)print("  WER=%.2f" % compute_wer(ref_en, hyp_en))

## §3 ASR经典流程传统ASR系统由四个模块组成：```语音信号 -> [特征提取] -> [声学模型] -> [语言模型] -> [解码器] -> 文本              |              |             |            |           MFCC/fbank     HMM/DNN       N-gram     Viterbi/Beam Search```### 3.1 特征提取- MFCC（梅尔频率倒谱系数）：模块3已学过- fbank（梅尔滤波器组输出）：更常用于现代ASR- Whisper使用的是log-Mel语谱图### 3.2 声学模型- 传统：GMM-HMM- 深度学习：DNN-HMM、CTC、Seq2Seq- 输入：声学特征序列- 输出：每个时间帧对应哪个音素/字符的概率### 3.3 语言模型- 建模文本序列的概率 P(y)- 传统：N-gram、RNN-LM- 现代：Transformer-LM- 作用：纠正声学模型的错误（如"天气"vs"天汽"）### 3.4 解码器- 搜索最优文本序列- Viterbi解码、Beam Search- 综合声学模型和语言模型的分数

## §4 CTC原理**CTC解决的核心问题**：输入帧和输出字符之间的对齐关系不确定。- 语音输入：每10ms一帧 → 1秒语音=100帧- 文本输出："你好" 只有2个字- 100个帧 → 2个字，如何对齐？**CTC的解决方案**：1. 引入空白符（blank），表示"这一帧没有输出"2. 允许重复字符（"nniiihhhaaooo" → "你好"）3. 对所有可能的对齐方式求边缘概率**CTC损失函数**：- P(l|x) = Σ_{π∈B^{-1}(l)} Π_t P(π_t|x)- 对所有能折叠成目标文本的路径求和> **直觉**：CTC不需要知道每个字在哪一帧，它自动学习对齐。

In [ ]:
# CTC 对齐演示import numpy as np# 模拟一个简单的CTC输出# 假设词汇表 = {blank, 你, 好, 呀}vocab = ['blank', '你', '好', '呀']T = 8  # 8个时间帧np.random.seed(42)# 模拟每帧的输出概率（已softmax）logits = np.random.randn(T, len(vocab))probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)print("=== CTC输出概率 ===")print("时间帧 -> 各字符概率 [blank, 你, 好, 呀]")for t in range(T):    pred = vocab[np.argmax(probs[t])]    print("  帧%d: %s -> 预测: %s" % (t, np.round(probs[t], 2), pred))print()# CTC解码：贪婪搜索（每帧取最大概率，然后折叠）path = [vocab[np.argmax(probs[t])] for t in range(T)]print("原始路径:", path)# 折叠：合并重复，去掉blankcollapsed = []prev = Nonefor s in path:    if s != prev and s != 'blank':        collapsed.append(s)    prev = sprint("折叠结果:", collapsed)print()print("这就是CTC的贪婪解码：先取每帧最大概率的输出，再合并重复和空白。")

## §5 Attention-based Seq2Seq + Transformer### Seq2Seq基本思想- **Encoder**：将输入序列编码为一个上下文向量- **Decoder**：逐个生成输出字符，每步关注输入的不同部分- **Attention**：让解码器"看"输入序列中与当前输出最相关的部分### Transformer在ASR中的应用- 用自注意力（Self-Attention）替代RNN- 并行计算，训练更快- Whisper就是基于Transformer的编码器-解码器架构### 与模块知识的对应| ASR概念 | 已学知识 ||---------|----------|| Encoder（编码器） | 模块2-3的CNN/CRNN特征提取 || Decoder（解码器） | 模块3的FC分类层 || Attention（注意力） | 模块4的SE（通道注意力） || CTC损失 | 模块2的交叉熵损失的变体 || Beam Search | 贪婪搜索的改进版 |

## §6 Whisper架构### 6.1 Whisper简介OpenAI于2022年发布的语音识别模型，特点：- **多语言**：支持99种语言，包括中文- **多任务**：语音识别、语音翻译、语言识别、语音活动检测- **大规模训练**：68万小时多语言音频- **Transformer架构**：编码器-解码器### 6.2 Whisper架构```输入音频 (16kHz)    |log-Mel语谱图 (80 bands, 30s窗口)    |Encoder (Transformer layers)    |编码特征    |Decoder (Transformer layers, 自回归)    |输出文本 token 序列```### 6.3 Whisper的特殊设计- **30秒窗口**：固定处理30秒音频段- **多任务token**：`<|zh|>`表示中文，`<|transcribe|>`表示识别- **无外部语言模型**：语言知识已内置在模型中- **鲁棒性**：在噪声环境下表现较好

In [ ]:
# Whisper 初步使用import os# 检查 whisper 是否安装try:    import whisper    print("Whisper 已安装，版本:", whisper.__version__ if hasattr(whisper, '__version__') else "未知")    WHISPER_AVAILABLE = Trueexcept ImportError:    print("Whisper 未安装")    print("安装方式: pip install openai-whisper")    print("或使用更快版本: pip install faster-whisper")    WHISPER_AVAILABLE = False# 检查 faster-whispertry:    from faster_whisper import WhisperModel    print("faster-whisper 已安装")    FASTER_AVAILABLE = Trueexcept ImportError:    FASTER_AVAILABLE = Falseprint()if WHISPER_AVAILABLE or FASTER_AVAILABLE:    print("ASR环境就绪！")else:    print("请先安装Whisper: pip install openai-whisper")

## §7 ASR与CI研究的关系### ASR作为CI评估工具ASR可以作为CI语音处理pipeline的**客观评估终点**：```带噪语音 → [语音增强] → [CI编码] → [ASR识别] → 计算CER```对比不同处理方式下的ASR准确率：| 处理方式 | 预期CER | 说明 ||---------|---------|------|| 干净语音直接识别 | 低（基线） | ASR模型本身的能力上限 || 带噪语音直接识别 | 高 | 噪声严重影响识别 || 增强后识别 | 中 | 语音增强应降低CER || 增强→ACE→识别 | 中-高 | ACE编码会损失信息 |### 局限性- ASR准确率 ≠ CI用户听懂程度- ASR模型训练在正常听力数据上，可能不反映CI用户的感知- 但ASR提供了一个可重复、可比较的客观指标> **思考题**：如果你要设计一个实验来评估"语音增强对CI用户的帮助"，你会用ASR还是主观测试？各有什么优缺点？

---## 小结本节课我们：1. 理解了ASR的问题定义和评估指标（WER/CER）2. 了解了ASR的经典流程：特征提取→声学模型→语言模型→解码3. 理解了CTC解决对齐问题的直觉4. 了解了Attention-based和Transformer架构5. 学习了Whisper的架构和使用方法6. 讨论了ASR与CI研究的关系**课后任务**：- 安装Whisper并成功加载模型- 用Whisper识别一段中文语音- 预习下次课：准备好前几个模块的输出结果